In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import chronopack as cp
from chronopack.tl import _gene_dynamics as gd
from chronopack.preprocessing import filter_low_count_genes, filter_technical_genes


In [2]:
adata_hvg = sc.read_h5ad('../output/adata_hvg.chk1.h5ad')
adata = sc.read_h5ad('../output/adata_full.chk1.h5ad')

# Compute CPM & logCPM matrices for negative bulk control

In [4]:
mtx_agg, cpm_ref, logcpm_ref = gd.load_negative_control_logcpm('../intmd/k562_neg-ctrl_mtx-agg.pkl')


# Compute pseudobulk CPM & logCPM matrices for LPS+THP1 experiment

In [6]:
cpm, logcpm = gd.temporal_pseudobulk_from_anndata(adata)


# Prioritize genes by variation over time

In [8]:
def RenameClusters(x):
    if (x == '0'): return 'K562'
    if (x == '1'): return 'K562'
    if (x == '2'): return 'THP1'

adata.obs['cell_type'] = adata.obs['traj_clusters'].apply(RenameClusters)

# New approach

In [10]:
adata_thp1_hvg = adata[(adata.obs['cell_type'] == 'THP1'), :].copy()
adata_k562_hvg = adata[(adata.obs['cell_type'] == 'K562'), :].copy()

In [11]:
cpm_thp1, logcpm_thp1 = gd.temporal_pseudobulk_from_anndata(adata_thp1_hvg)
cpm_k562, logcpm_k562 = gd.temporal_pseudobulk_from_anndata(adata_k562_hvg)

In [55]:
adata_k562_hvg.var["total_counts"] = np.asarray(
    adata_k562_hvg.layers["counts"].sum(axis=0)
).ravel()

In [57]:
adata_thp1_hvg.var["total_counts"] = np.asarray(
    adata_thp1_hvg.layers["counts"].sum(axis=0)
).ravel()

In [ ]:
low_count_genes_k562 = set(adata_k562_hvg.var.index[adata_k562_hvg.var["total_counts"] < 10])
low_count_genes_thp1 = set(adata_thp1_hvg.var.index[adata_thp1_hvg.var["total_counts"] < 10])
print(f"K562 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_k562)}")
print(f"THP1 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_thp1)}")


In [13]:
adata_hvg = adata.copy()
shared_genes = list(set(adata_hvg.var.index) & set(logcpm_ref.columns) & set(logcpm.columns))

cpm_ref = cpm_ref.loc[:, shared_genes]
logcpm_ref = logcpm_ref.loc[:, shared_genes]

cpm = cpm.loc[:, shared_genes]
logcpm = logcpm.loc[:, shared_genes]
logcpm.shape

(12, 19433)

# Compute correlations

In [14]:
corr_k562 = gd.compute_smooth_correlations(
    logcpm_k562,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)

corr_thp1 = gd.compute_smooth_correlations(
    logcpm_thp1,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.7,
    local_fraction_threshold=0.3,
)


KeyboardInterrupt: 

In [ ]:
corr_k562 = gd.refilter_tag_bias_candidates(
    corr_k562,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_k562["tag_bias_candidate"].sum()))

corr_k562[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


In [ ]:
corr_thp1 = gd.refilter_tag_bias_candidates(
    corr_thp1,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_thp1["tag_bias_candidate"].sum()))

corr_thp1[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


In [ ]:
excl_genes_k562 = gd.tag_bias_exclusion_genes(corr_k562)
print(len(excl_genes_k562))
print('HNRNPA1P27' in excl_genes_k562)

In [ ]:
excl_genes_thp1 = gd.tag_bias_exclusion_genes(corr_thp1)
print(len(excl_genes_thp1))
print('HNRNPA1P27' in excl_genes_thp1)

# Pre-filter @K562

In [ ]:
adata_k562_hvg = filter_technical_genes(adata_k562_hvg, exclude_genes=excl_genes_k562)
adata_k562_hvg = filter_low_count_genes(adata_k562_hvg, min_total_counts=10, counts_layer="counts")
adata_k562_hvg


In [ ]:
print('PLAU' in set(adata_k562_hvg.var.index)) # similar shape
print('CD83' in set(adata_k562_hvg.var.index)) # similar shape
print('EGR1' in set(adata_k562_hvg.var.index)) # similar shape
print('IER3' in set(adata_k562_hvg.var.index)) # similar shape

In [ ]:
# TPs
print('REL' in set(adata_k562_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_k562_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_k562_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_k562_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_k562_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_k562_hvg.var.index)) # similar shape

# Pre-filter @THP1

In [ ]:
adata_thp1_hvg = filter_technical_genes(adata_thp1_hvg, exclude_genes=excl_genes_thp1)
adata_thp1_hvg = filter_low_count_genes(adata_thp1_hvg, min_total_counts=10, counts_layer="counts")
adata_thp1_hvg


In [ ]:
print('PLAU' in set(adata_thp1_hvg.var.index)) # similar shape
print('CD83' in set(adata_thp1_hvg.var.index)) # similar shape
print('EGR1' in set(adata_thp1_hvg.var.index)) # similar shape
print('IER3' in set(adata_thp1_hvg.var.index)) # similar shape

In [ ]:
# TPs
print('REL' in set(adata_thp1_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_thp1_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_thp1_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_thp1_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_thp1_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_thp1_hvg.var.index)) # similar shape

In [ ]:
print('HNRNPA1P27' in adata_k562_hvg.var.index)
print('HNRNPA1P27' in adata_thp1_hvg.var.index)

# Prepare input files for TradeSeq (per cell type)

In [ ]:
adata_full = sc.read_h5ad('../output/adata_full.chk1.h5ad')
adata_k562_full = adata_full[adata_k562_hvg.obs.index, :]
print(adata_k562_full.shape)

adata_thp1_full = adata_full[adata_thp1_hvg.obs.index, :]
print(adata_thp1_full.shape)

# Prepare input files for TradeSeq (across cell types)

In [ ]:
adata_k562_hvg.write_h5ad('../output/adata_k562_prefiltered.fixed.h5ad')
adata_thp1_hvg.write_h5ad('../output/adata_thp1_prefiltered.fixed.h5ad')


In [ ]:
cp.tl.fit_gene_dynamics(
    adata_k562_hvg,
    output_dir='../intmd/sc-K562_to_tradeSeq.fixed',
    full_adata=adata_k562_full,
    prepare_inputs=True,
    run_backend=False,
)
cp.tl.fit_gene_dynamics(
    adata_thp1_hvg,
    output_dir='../intmd/sc-THP1_to_tradeSeq.fixed',
    full_adata=adata_thp1_full,
    prepare_inputs=True,
    run_backend=False,
)


In [ ]:
# Full-library tradeSeq inputs are written by the package-backed calls above.
